<div style="background-color: #0b1f1a; padding: 25px; border-left: 10px solid #00f59f; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
    <h1 style="color: #ffffff; margin: 0; font-family: 'Segoe UI', Arial, sans-serif; font-size: 32px;">
        📊 Overview Page Validation
    </h1>
    <p style="color: #b0c9c2; margin: 10px 0 0 0; font-size: 16px; font-family: 'Segoe UI', Arial, sans-serif;">
        Automated SQL Data Extraction & Source-of-Truth Verification Pipeline
    </p>
</div>

In [4]:
import os
import boto3
import duckdb
import pandas as pd
import dataframe_image as dfi
from io import BytesIO
from dotenv import load_dotenv

# 1. Load AWS Credentials from .env
load_dotenv(dotenv_path="../../.env")
AWS_REGION = os.getenv("AWS_REGION")
PROCESSED_BUCKET = 'ra2-processed-data'

# 2. Setup overview_output folder
output_folder = "../../eda_outputs/overview_output"
os.makedirs(output_folder, exist_ok=True)

# 3. Connect to S3 securely
s3 = boto3.client('s3', region_name=AWS_REGION)
print("🔗 Connecting to S3 Processed Bucket...\n")

# 4. Download ALL files from S3 and explicitly register them in DuckDB
response = s3.list_objects_v2(Bucket=PROCESSED_BUCKET)

print("📥 Downloading and Registering Clean Tables:")
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    if file_key.endswith('.csv'):
        table_name = file_key.replace('.csv', '').lower()
        print(f"   -> Fetching {file_key} and registering as table: '{table_name}'...")
        
        csv_obj = s3.get_object(Bucket=PROCESSED_BUCKET, Key=file_key)
        df_temp = pd.read_csv(BytesIO(csv_obj['Body'].read()))
        
        # Explicitly register the dataframe into DuckDB
        duckdb.register(table_name, df_temp)

print("\n✅ All clean data successfully registered in DuckDB!")

# 5. DuckDB Validation Function
def save_sql_to_image(sql_query, image_name):
    print(f"Executing query for: {image_name}...")
    df = duckdb.query(sql_query).df()
    
    file_path = os.path.join(output_folder, f"{image_name}.png")
    dfi.export(df, file_path, table_conversion="matplotlib")
    
    print(f"✅ Image saved successfully at: {file_path}\n")
    return df

🔗 Connecting to S3 Processed Bucket...

📥 Downloading and Registering Clean Tables:
   -> Fetching customers.csv and registering as table: 'customers'...
   -> Fetching delivery.csv and registering as table: 'delivery'...
   -> Fetching feedback.csv and registering as table: 'feedback'...
   -> Fetching menu_items.csv and registering as table: 'menu_items'...
   -> Fetching order_items.csv and registering as table: 'order_items'...
   -> Fetching orders.csv and registering as table: 'orders'...
   -> Fetching outlets.csv and registering as table: 'outlets'...
   -> Fetching promotions.csv and registering as table: 'promotions'...

✅ All clean data successfully registered in DuckDB!


In [5]:
# KPI 1: Total Revenue (From S3 Processed Data)
kpi_revenue_sql = """
SELECT 
    ROUND(SUM(final_amount) / 1000000, 2) AS total_revenue_millions
FROM orders
WHERE order_status = 'Delivered';
"""

save_sql_to_image(kpi_revenue_sql, "1_kpi_total_revenue")

Executing query for: 1_kpi_total_revenue...
✅ Image saved successfully at: ../../eda_outputs/overview_output\1_kpi_total_revenue.png



,total_revenue_millions
0,51.8


In [6]:
# KPI 2: Total Orders
kpi_orders_sql = """
SELECT 
    ROUND(COUNT(order_id) / 1000, 2) AS total_orders_k
FROM orders;
"""

save_sql_to_image(kpi_orders_sql, "2_kpi_total_orders")

Executing query for: 2_kpi_total_orders...
✅ Image saved successfully at: ../../eda_outputs/overview_output\2_kpi_total_orders.png



,total_orders_k
0,79.81


In [7]:
# KPI 3: Total Customers
kpi_customers_sql = """
SELECT 
    ROUND(COUNT(customer_id) / 1000, 2) AS total_customers_k
FROM customers;
"""

save_sql_to_image(kpi_customers_sql, "3_kpi_total_customers")

Executing query for: 3_kpi_total_customers...
✅ Image saved successfully at: ../../eda_outputs/overview_output\3_kpi_total_customers.png



,total_customers_k
0,15.25


In [8]:
# KPI 4: YoY Growth Percentage (Based on Delivered Orders)
kpi_yoy_growth_sql = """
WITH yearly_revenue AS (
    SELECT 
        YEAR(CAST(order_date AS DATE)) AS order_year,
        SUM(final_amount) AS total_revenue
    FROM orders
    WHERE order_status = 'Delivered'
    GROUP BY YEAR(CAST(order_date AS DATE))
)
SELECT 
    t1.order_year AS current_year,
    ROUND(t1.total_revenue / 1000000, 2) AS current_rev_m,
    ROUND(t2.total_revenue / 1000000, 2) AS prev_rev_m,
    ROUND(((t1.total_revenue - t2.total_revenue) / t2.total_revenue) * 100, 2) AS yoy_growth_pct
FROM yearly_revenue t1
JOIN yearly_revenue t2 ON t1.order_year = t2.order_year + 1
ORDER BY t1.order_year DESC
LIMIT 1;
"""

save_sql_to_image(kpi_yoy_growth_sql, "4_kpi_yoy_growth")

Executing query for: 4_kpi_yoy_growth...
✅ Image saved successfully at: ../../eda_outputs/overview_output\4_kpi_yoy_growth.png



,current_year,current_rev_m,prev_rev_m,yoy_growth_pct
0,2027,0.09,8.85,-98.96


In [9]:
# KPI 5: Retention Rate (Translated directly from your Power BI DAX logic)
kpi_retention_sql = """
WITH customer_orders AS (
    SELECT 
        customer_id,
        COUNT(DISTINCT order_id) AS order_count
    FROM orders
    GROUP BY customer_id
),
metrics AS (
    SELECT 
        SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) AS repeat_customers,
        COUNT(customer_id) AS total_customers
    FROM customer_orders
)
SELECT 
    repeat_customers,
    total_customers,
    ROUND((CAST(repeat_customers AS DOUBLE) / total_customers) * 100, 2) AS retention_rate_pct
FROM metrics;
"""

save_sql_to_image(kpi_retention_sql, "5_kpi_retention_rate")

Executing query for: 5_kpi_retention_rate...
✅ Image saved successfully at: ../../eda_outputs/overview_output\5_kpi_retention_rate.png



,repeat_customers,total_customers,retention_rate_pct
0,10440.0,12856,81.21


In [11]:
import matplotlib.pyplot as plt
import os

def save_chart_to_image(df, x_col, y_col, title, image_name, xlabel="", ylabel=""):
    print(f"Generating visual chart for: {image_name}...")
    
    # Create figure with your dashboard's dark background color
    plt.figure(figsize=(10, 6), facecolor='#0b1f1a')
    ax = plt.axes()
    ax.set_facecolor('#0b1f1a')
    
    # Draw bars with your signature neon green color
    plt.bar(df[x_col], df[y_col], color='#00f59f', width=0.6)
    
    # Title and labels styling
    plt.title(title, color='white', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel(xlabel, color='#b0c9c2', fontsize=12)
    plt.ylabel(ylabel, color='#b0c9c2', fontsize=12)
    
    # Tick formatting
    plt.xticks(color='#b0c9c2', fontsize=10, rotation=15)
    plt.yticks(color='#b0c9c2', fontsize=10)
    
    # Grid lines matching dashboard style
    plt.grid(axis='y', linestyle='--', alpha=0.2, color='#b0c9c2')
    
    # Border styling
    for spine in ax.spines.values():
        spine.set_color('#2e5e52')
        
    plt.tight_layout()
    
    # Save as high-res PNG image
    file_path = os.path.join(output_folder, f"{image_name}.png")
    plt.savefig(file_path, facecolor='#0b1f1a', edgecolor='none', dpi=300)
    plt.close()
    
    print(f"📈 Chart saved successfully at: {file_path}\n")

In [12]:
# 1. Query Data for Total Revenue by City
chart_revenue_by_city_sql = """
SELECT 
    o.city,
    ROUND(SUM(ord.final_amount) / 1000000, 2) AS revenue_millions
FROM orders ord
JOIN outlets o ON ord.outlet_id = o.outlet_id
WHERE ord.order_status = 'Delivered'
GROUP BY o.city
ORDER BY revenue_millions DESC;
"""

df_city = duckdb.query(chart_revenue_by_city_sql).df()

# 2. Draw and Save as a Visual Chart Image
save_chart_to_image(
    df=df_city,
    x_col='city',
    y_col='revenue_millions',
    title='Total Revenue by City',
    image_name='6_chart_revenue_by_city',
    xlabel='City',
    ylabel='Revenue (₹ in Millions)'
)

Generating visual chart for: 6_chart_revenue_by_city...
📈 Chart saved successfully at: ../../eda_outputs/overview_output\6_chart_revenue_by_city.png



In [16]:
import matplotlib.pyplot as plt
import os

# 1. Query data for the 4 supportive charts using DuckDB
df_order_type = duckdb.query("""
    SELECT order_type, ROUND(SUM(final_amount) / 1000, 2) AS revenue_k
    FROM orders WHERE order_status = 'Delivered'
    GROUP BY order_type ORDER BY revenue_k DESC;
""").df()

# Using the correct column name: discount_applied
df_promo = duckdb.query("""
    SELECT p.promo_code, ROUND(SUM(o.discount_applied), 2) AS total_discount
    FROM orders o
    JOIN promotions p ON o.promo_id = p.promo_id
    WHERE o.order_status = 'Delivered' AND o.promo_id IS NOT NULL
    GROUP BY p.promo_code ORDER BY total_discount DESC;
""").df()

df_loyalty = duckdb.query("""
    SELECT loyalty_tier, COUNT(customer_id) AS customer_count
    FROM customers
    GROUP BY loyalty_tier ORDER BY customer_count DESC;
""").df()

df_delivery = duckdb.query("""
    SELECT delivery_partner, COUNT(order_id) AS total_deliveries
    FROM delivery
    GROUP BY delivery_partner ORDER BY total_deliveries DESC;
""").df()


# 2. Setup a 2x2 subplot grid matching your dashboard aesthetic
fig, axs = plt.subplots(2, 2, figsize=(14, 10), facecolor='#0b1f1a')
axs = axs.ravel()

charts_data = [
    (axs[0], df_order_type, 'order_type', 'revenue_k', 'Total Revenue by Order Type', 'Revenue (₹K)', 'bar'),
    (axs[1], df_promo, 'promo_code', 'total_discount', 'Discount Value by Promo Code', 'Discount (₹)', 'bar'),
    (axs[2], df_loyalty, 'loyalty_tier', 'customer_count', 'Loyalty Tier Distribution Count', 'Customer Count', 'barh'),
    (axs[3], df_delivery, 'delivery_partner', 'total_deliveries', 'Delivery Distribution by Partner', 'Deliveries', 'bar')
]

for ax, df, x_col, y_col, title, ylabel, chart_type in charts_data:
    ax.set_facecolor('#0b1f1a')
    
    if chart_type == 'bar':
        ax.bar(df[x_col], df[y_col], color='#00f59f', width=0.55)
        ax.tick_params(axis='x', rotation=15, colors='#b0c9c2')
    elif chart_type == 'barh':
        ax.barh(df[x_col], df[y_col], color='#00f59f', height=0.55)
        ax.invert_yaxis()  # Highest at the top
        ax.tick_params(axis='y', colors='#b0c9c2')

    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.tick_params(axis='y', colors='#b0c9c2')
    ax.grid(axis='y', linestyle='--', alpha=0.15, color='#b0c9c2')
    
    for spine in ax.spines.values():
        spine.set_color('#2e5e52')

plt.tight_layout()

# 3. Save the combined dashboard overview image
file_path = os.path.join(output_folder, "7_to_10_supportive_charts_overview.png")
plt.savefig(file_path, facecolor='#0b1f1a', edgecolor='none', dpi=300)
plt.close()

print(f"✅ All 4 supportive charts successfully generated and saved at: {file_path}")

✅ All 4 supportive charts successfully generated and saved at: ../../eda_outputs/overview_output\7_to_10_supportive_charts_overview.png
